# SpaceX Falcon 9 First Stage Landing Prediction

## Data Wrangling
This notebook prepares the raw SpaceX launch records for analysis by cleaning the landing outcomes, creating the binary target variable, and exporting a structured dataset for downstream notebooks.

## Contents
1. Load the raw launch data
2. Explore launch sites, orbits, and outcomes
3. Create the binary landing label
4. Export the cleaned dataset

## 1. Load Required Libraries
Import the libraries needed for data manipulation and analysis.

In [ ]:
import pandas as pd
import numpy as np

## 2. Load the Raw Dataset
Load the SpaceX launch data from the raw data folder and inspect the first few rows.

In [ ]:
df = pd.read_csv('../data/raw/spacex_launch_data.csv')
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,6,2010-06-04,Falcon 9,6123.547647,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,8,2012-05-22,Falcon 9,525.000000,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,10,2013-03-01,Falcon 9,677.000000,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,11,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,12,2013-12-03,Falcon 9,3170.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 3. Explore Launch Sites
Review the distribution of launches across launch sites.

In [ ]:
launch_site_counts = df['LaunchSite'].value_counts()
launch_site_counts

LaunchSite
CCSFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

## 4. Explore Orbits
Review the distribution of launch orbits.

In [ ]:
orbit_counts = df['Orbit'].value_counts()
orbit_counts

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64

## 5. Explore Landing Outcomes
Review the recorded landing outcomes to determine which ones count as failures.

In [ ]:
landing_outcomes = df['Outcome'].value_counts()
landing_outcomes

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

### 5.1 Inspect Each Outcome
Display each landing outcome with its index to identify the failure labels.

In [ ]:
for index, outcome in enumerate(landing_outcomes.keys()):
    print(index, outcome)

0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


### 5.2 Define Failed Outcomes
Create the set of outcomes that correspond to failed landings.

In [ ]:
bad_outcomes = set(landing_outcomes.keys()[[1, 3, 5, 6, 7]])
bad_outcomes

{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

## 6. Create the Binary Target
Create the `Class` column used for later machine learning models.

In [ ]:
df['Class'] = df['Outcome'].apply(lambda x: 0 if x in bad_outcomes else 1)
landing_class = df['Class'].values

### 6.1 Verify the Target Column
Display the first few rows of the `Class` column to confirm the labels were created correctly.

In [ ]:
df['Class'] = landing_class

df[['Class']].head(8)

,Class
0,0
1,0
2,0
3,0
4,0
5,0
6,1
7,1


### 6.2 Review the Full Dataset
Inspect the full cleaned dataset with the new target column included.

In [ ]:
df.head(10)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,6,2010-06-04,Falcon 9,6123.547647,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,8,2012-05-22,Falcon 9,525.000000,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,10,2013-03-01,Falcon 9,677.000000,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,11,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,12,2013-12-03,Falcon 9,3170.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0
5,13,2014-01-06,Falcon 9,3325.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857,0
6,14,2014-04-18,Falcon 9,2296.000000,ISS,CCSFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857,1
7,15,2014-07-14,Falcon 9,1316.000000,LEO,CCSFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857,1
8,16,2014-08-05,Falcon 9,4535.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857,0
9,17,2014-09-07,Falcon 9,4428.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857,0


## 7. Export the Cleaned Dataset
Save the cleaned dataframe to the processed data folder for downstream analysis.

In [ ]:
df.to_csv('../data/processed/spacex_launch_data_clean.csv', index=False)
print("Data successfully saved to '../data/processed/spacex_launch_data_clean.csv'")

Data successfully saved to 'spacex_launch_data_clean.csv'


## Summary
This notebook prepared the raw SpaceX launch data for later analysis by:

- Loading the launch records from the raw data folder
- Reviewing launch site, orbit, and landing outcome distributions
- Creating a binary landing label for model training
- Exporting the cleaned dataset to the processed data folder

The output is ready for the EDA and machine learning notebooks.